Подключение Google Disk, создание итоговой директории, куда положить csv таблицу:

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
# Заменить путь нужный
drive_results_path = "/content/drive/MyDrive/pyannote_results"
os.makedirs(drive_results_path, exist_ok=True)

Скачиваем необходимые зависимости:

In [ ]:
!apt-get update && apt-get install -y ffmpeg

!pip install --no-cache-dir \
    "pyannote.audio" \
    "pydub" \
    "librosa" \
    "datasets" \
    "huggingface_hub"

Определяем девайс, на котором будет исполняться сложный код: CPU, GPU (cuda)

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Selected device: {device}")

Обязательно авторизируемся в Hugging Face Hub для возможности скачивания модели диаризации:

In [ ]:
from google.colab import userdata
from huggingface_hub import login
import os

try:
    print("Logging in HuggingFace...")
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    print("Token has downloaded from Colab's secrets!")
    login(token=hf_token)
    print("Login successfully!")
except Exception as e:
    print(f"Error: {e}")
    hf_token = None

Определяем допуски для метрики DER:

In [ ]:
from pyannote.metrics.diarization import DiarizationErrorRate, DiarizationPurity, DiarizationCoverage
from pyannote.metrics.detection import DetectionErrorRate

configs = {
    'strict': {'collar': 0.0, 'skip_overlap': False},
    'collar': {'collar': 0.25, 'skip_overlap': False},
    'no_ovl': {'collar': 0.0, 'skip_overlap': True},
    'clean':  {'collar': 0.25, 'skip_overlap': True}
}

metrics_vault = {}
for c_name, params in configs.items():
    metrics_vault[c_name] = {
        'der': DiarizationErrorRate(**params),
        'purity': DiarizationPurity(**params),
        'coverage': DiarizationCoverage(**params),
        'det': DetectionErrorRate(collar=params['collar'])
    }

На тестовых данных из датасета AMI (ihm) тестируем базовый пайплайн Pyannote. В конце сохраняем все данные в csv таблицу, которая будет состоять из общей ошибки DER, трех ее составляющих: Missed, False alarm, Confusion и из метрик Purity и Coverage.

In [ ]:
from datasets import load_dataset
from datetime import datetime
import os
from pyannote.core import Annotation, Segment
from pyannote.audio import Pipeline
import pandas as pd
import soundfile as sf
import torch

diarization_data_set = "diarizers-community/ami"
dataset_config = "ihm" # default: None

os.makedirs("dataset_samples", exist_ok=True)
os.makedirs("results", exist_ok=True)

pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1")
pipeline.to(torch.device(device))
dataset = load_dataset(f"{diarization_data_set}", name=dataset_config, split="test", streaming=True)

test_samples = list(dataset)
results = []
for idx, sample in enumerate(test_samples):
    audio = sample["audio"]
    audio_path = f"dataset_samples/sample_{idx}.wav"
    sf.write(audio_path, audio["array"], audio["sampling_rate"])

    reference = Annotation()
    for start, end, speaker in zip(sample['timestamps_start'], sample['timestamps_end'], sample['speakers']):
        segment = Segment(start, end)
        reference[segment] = speaker

    hypothesis = pipeline(audio_path)
    hypothesis_diarization = hypothesis.speaker_diarization

    res = {'file': f"sample_{idx}"}
    for c_name, m_group in metrics_vault.items():
        res[f'DER_{c_name}'] = m_group['der'](reference, hypothesis_diarization)
        res[f'Purity_{c_name}'] = m_group['purity'](reference, hypothesis_diarization)
        res[f'Coverage_{c_name}'] = m_group['coverage'](reference, hypothesis_diarization)
        res[f'DetER_{c_name}'] = m_group['det'](reference, hypothesis_diarization)

        components = m_group['der'].compute_components(reference, hypothesis_diarization)
        res[f'FA_{c_name}'] = components['false alarm']
        res[f'Miss_{c_name}'] = components['missed detection']
        res[f'Conf_{c_name}'] = components['confusion']
        res[f'Total_Speech_{c_name}'] = components['total']

        print("-" * 60)
        print("DER: ", res[f'DER_{c_name}'])
        print("Purity: ", res[f'Purity_{c_name}'])
        print("Coverage: ", res[f'Coverage_{c_name}'])
        print("DetER: ", res[f'DetER_{c_name}'])
        print(f"- {c_name} -")
        print("FA: ", res[f'FA_{c_name}'])
        print("Miss:", res[f'Miss_{c_name}'])
        print("Conf: ", res[f'Conf_{c_name}'])
    print("================================================== New Test audio file ==================================================")

    results.append(res)

current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
df = pd.DataFrame(results)
df.to_csv(f"results/pyannote_metrics_{current_time}.csv", index=False)
if not os.path.exists(f"{drive_results_path}"):
    os.makedirs(f"{drive_results_path}/")
    df.to_csv(f"{drive_results_path}/pyannote_metrics_{current_time}.csv", index=False)

Визуализируем результаты: общие (средняя ошибка DER) и для каждой записи. Здесь визуализируется DER из 3 частей и отдельным графиком Purity и Coverage.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Заменить путь нужный. Это путь до итогового файла csv после тестирования
df = pd.read_csv("/content/drive/MyDrive/pyannote_results/pyannote_metrics_20260323_133442.csv")

res = df.mean(numeric_only=True).to_dict()
print(res)

visualize_configs = ["strict", "collar", "no_ovl", "clean"]
for c_name in visualize_configs:
    der = res[f'DER_{c_name}']
    total = res[f'Total_Speech_{c_name}']
    fa = (res[f'FA_{c_name}'] / total) * 100
    miss = (res[f'Miss_{c_name}'] / total) * 100
    conf = (res[f'Conf_{c_name}'] / total) * 100
    purity = res[f'Purity_{c_name}'] * 100
    coverage = res[f'Coverage_{c_name}'] * 100

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
    fig.suptitle(f'Configuration: {c_name.upper()}, DER: {round(der * 100, 3)}%', fontsize=18, fontweight='bold', y=1.02)

    # Graph 1. Components of DER
    labels = ['Missed', 'False Alarm', 'Confusion']
    errors = [miss, fa, conf]
    colors = ['#FF6B6B', '#4D96FF', '#6BCB77']

    bars1 = ax1.bar(labels, errors, color=colors, edgecolor='black', alpha=0.8)
    max_err = max(errors)
    upper_limit = max_err * 1.15 if max_err > 0 else 10

    ax1.set_ylim(0, upper_limit)
    ax1.set_title(f'DER Components', fontsize=14, pad=20)
    ax1.set_ylabel('Error Rate (%)')
    ax1.grid(axis='y', linestyle='--', alpha=0.6)

    for bar in bars1:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width() / 2., height + (upper_limit * 0.01),
                f'{height:.2f}%', ha='center', va='bottom',
                fontweight='bold', fontsize=11, color='black')

    # Graph 2. Purity & Coverage Quality
    q_labels = ['Purity', 'Coverage']
    q_values = [purity, coverage]
    q_colors = ['#FFD93D', '#A084CA']

    bars2 = ax2.bar(q_labels, q_values, color=q_colors, edgecolor='black', width=0.5)

    ax2.set_ylim(0, 105)
    ax2.set_title('Quality Metrics (Higher is Better)', fontsize=14, pad=20)
    ax2.set_ylabel('Score (%)')
    ax2.grid(axis='y', linestyle='--', alpha=0.3)

    for bar in bars2:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width() / 2., height + 1,
                f'{height:.1f}%', ha='center', va='bottom',
                fontweight='bold', fontsize=11)

    plt.tight_layout()
    plt.savefig(f'der_components_{c_name}.png', dpi=300, bbox_inches='tight')
    plt.show()
    print()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


visualize_configs = ["strict", "collar", "no_ovl", "clean"]
for c_name in visualize_configs:
    total = res[f'Total_Speech_{c_name}']
    fa = (res[f'FA_{c_name}'] / total) * 100
    miss = (res[f'Miss_{c_name}'] / total) * 100
    conf = (res[f'Conf_{c_name}'] / total) * 100
    purity = res[f'Purity_{c_name}'] * 100
    coverage = res[f'Coverage_{c_name}'] * 100

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
    fig.suptitle(f'Configuration: {c_name.upper()}', fontsize=18, fontweight='bold', y=1.02)

    # Graph 1. Components of DER
    labels = ['Missed', 'False Alarm', 'Confusion']
    errors = [miss, fa, conf]
    colors = ['#FF6B6B', '#4D96FF', '#6BCB77']

    bars1 = ax1.bar(labels, errors, color=colors, edgecolor='black', alpha=0.8)
    max_err = max(errors)
    upper_limit = max_err * 1.15 if max_err > 0 else 10

    ax1.set_ylim(0, upper_limit)
    ax1.set_title(f'DER Components', fontsize=14, pad=20)
    ax1.set_ylabel('Error Rate (%)')
    ax1.grid(axis='y', linestyle='--', alpha=0.6)

    for bar in bars1:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + (upper_limit * 0.01),
                f'{height:.2f}%', ha='center', va='bottom',
                fontweight='bold', fontsize=11, color='black')

    # Graph 2. Purity & Coverage Quality
    q_labels = ['Purity', 'Coverage']
    q_values = [purity, coverage]
    q_colors = ['#FFD93D', '#A084CA']

    bars2 = ax2.bar(q_labels, q_values, color=q_colors, edgecolor='black', width=0.5)

    ax2.set_ylim(0, 105)
    ax2.set_title('Quality Metrics (Higher is Better)', fontsize=14, pad=20)
    ax2.set_ylabel('Score (%)')
    ax2.grid(axis='y', linestyle='--', alpha=0.3)

    for bar in bars2:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{height:.1f}%', ha='center', va='bottom',
                fontweight='bold', fontsize=11)

    plt.tight_layout()
    plt.show()
    print()